# Recall@K on a random sample of jobs

This notebook randomly selects `X` jobs, retrieves candidates using exact cosine similarity, and measures how many known relevant candidates appear within several top-K cutoffs. Exact search is used so this notebook measures the embedding representation rather than HNSW approximation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'matching-pipeline').exists()), None)
if ROOT is None:
    raise RuntimeError('Could not locate the repository root.')
sys.path.insert(0, str(ROOT / 'matching-pipeline/retrieval/src'))

from retrieval.embeddings import embedding_matrix
from retrieval.evaluation import evaluate_retrieval
from retrieval.exact_search import exact_cosine_search

## Parameters

Change `X` to evaluate a larger or smaller random job sample. The seed makes the sample reproducible. Grades 1 and 2 both count as relevant in the primary evaluation.

In [ ]:
X = 100
RANDOM_SEED = 20260911
K_VALUES = [10, 25, 50, 100, 250, 500]
MINIMUM_RELEVANCE_GRADE = 1

EMBEDDING_DIR = ROOT / 'matching-pipeline/retrieval/artifacts/embeddings'
LABEL_PATH = ROOT / 'ground-truth-generation/data/v1/relevance_ground_truth.parquet'

## Load embeddings and select jobs

Every sampled job is compared with the complete currently embedded candidate pool.

In [ ]:
candidates = pd.read_parquet(EMBEDDING_DIR / 'candidates.parquet').sort_values('entity_id').reset_index(drop=True)
all_jobs = pd.read_parquet(EMBEDDING_DIR / 'jobs.parquet').sort_values('entity_id').reset_index(drop=True)

if X <= 0:
    raise ValueError('X must be positive.')
sample_size = min(X, len(all_jobs))
jobs = all_jobs.sample(n=sample_size, random_state=RANDOM_SEED).sort_values('entity_id').reset_index(drop=True)

print(f'Candidates searched: {len(candidates):,}')
print(f'Jobs sampled: {len(jobs):,} of {len(all_jobs):,}')
print(f'Embedding model: {candidates.model.iloc[0]}')
print(f'Vector dimensions: {int(candidates.dimensions.iloc[0]):,}')

## Retrieve candidates

The result contains one ranked candidate list per sampled job, up to the largest requested K.

In [ ]:
candidate_matrix = embedding_matrix(candidates)
job_matrix = embedding_matrix(jobs)
indices, similarities = exact_cosine_search(job_matrix, candidate_matrix, max(K_VALUES))

candidate_ids = candidates.entity_id.to_numpy()
retrieval_rows = []
for job_row, job_id in enumerate(jobs.entity_id):
    for rank, (candidate_row, similarity) in enumerate(zip(indices[job_row], similarities[job_row], strict=True), start=1):
        retrieval_rows.append({
            'job_id': job_id,
            'candidate_id': candidate_ids[candidate_row],
            'rank': rank,
            'cosine_similarity': float(similarity),
        })
retrieval = pd.DataFrame(retrieval_rows)
retrieval.head()

## Calculate Recall@K

A candidate is relevant when `relevance_grade >= 1`. Jobs with no known relevant candidates are reported separately and excluded from recall averages.

In [ ]:
relevance = pd.read_parquet(
    LABEL_PATH,
    columns=['job_id', 'candidate_id', 'relevance_grade'],
    filters=[('job_id', 'in', jobs.entity_id.tolist())],
)
per_job, summary = evaluate_retrieval(
    retrieval, relevance, K_VALUES, minimum_grade=MINIMUM_RELEVANCE_GRADE
)

summary_table = pd.DataFrame.from_dict(summary['cutoffs'], orient='index')
summary_table.index = summary_table.index.astype(int)
summary_table.index.name = 'K'
summary_table.sort_index().style.format('{:.3f}')

In [ ]:
print(f"Evaluable jobs: {summary['evaluable_jobs']}")
print(f"Jobs with zero known positives: {summary['zero_positive_jobs']}")
per_job.head()

## Graph Recall@K

The line is macro recall: every job contributes equally. The shaded band shows the 10th through 90th percentile across jobs, making weak-job behavior visible instead of showing only an average.

In [ ]:
evaluable = per_job.loc[per_job.relevant_count > 0]
macro = [evaluable[f'recall_at_{k}'].mean() for k in K_VALUES]
p10 = [evaluable[f'recall_at_{k}'].quantile(0.10) for k in K_VALUES]
p90 = [evaluable[f'recall_at_{k}'].quantile(0.90) for k in K_VALUES]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(K_VALUES, macro, marker='o', linewidth=2.5, label='Macro Recall@K')
ax.fill_between(K_VALUES, p10, p90, alpha=0.2, label='10th–90th percentile across jobs')
for k, value in zip(K_VALUES, macro, strict=True):
    ax.annotate(f'{value:.1%}', (k, value), xytext=(0, 8), textcoords='offset points', ha='center')
ax.set(title=f'Retrieval recall on {len(jobs)} randomly sampled jobs', xlabel='K candidates retrieved', ylabel='Recall@K', ylim=(0, 1.05))
ax.grid(alpha=0.25)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Interpretation

Recall should increase as K increases because a larger shortlist has more chances to contain known relevant candidates. The important tradeoff is choosing a K large enough to preserve strong candidates while remaining small enough for the later ranking model to score economically.